In [ ]:
import xml.etree.ElementTree as ET
import glob
import pandas as pd
import numpy as np

In [ ]:
excel_path = r"./Verlauf.xlsx"

In [ ]:
df_capacity = pd.read_excel(excel_path,sheet_name="Kapazität",header=2,skiprows=[3],usecols="B:Q")
df_capacity

In [ ]:
df_power = pd.read_excel(excel_path,sheet_name="Leistungen",header=[2,3])
df_power.drop(columns=["Unnamed: 0_level_0"],inplace=True)
df_power

In [ ]:
df_dyn_cap = pd.read_excel(excel_path,sheet_name="Dynamische Kapazität 25°C",header=[2,3])
df_dyn_cap.drop(columns=["Unnamed: 0_level_0"],inplace=True)
df_dyn_cap

In [ ]:
df_efc = pd.read_excel(excel_path,sheet_name="Zyklenanzahl",header=2,skiprows=[3],usecols="B:Q")
df_efc

In [ ]:
measurements = glob.glob("./**/*.irf", recursive=True)
measurements

In [ ]:
def extract_impedance_spectrum_to_df(measurement, id, df_capacity, df_power, df_dyn_cap, df_efc):
    checkup = measurement.split("\\")[1]
    temp = float(measurement.split("\\")[2].split("C")[0])
    soc = float(measurement.split("\\")[3].split("%")[0])/100
    cell_name = measurement.split("\\")[4].split("_")[1]

    cell_name = "Cell "+cell_name.split("NR")[1]

    capacity = df_capacity.loc[df_capacity['Cell-Name'] == checkup,cell_name].values[0]
    efc = df_efc.loc[df_efc['Cell-Name'] == checkup,cell_name].values[0]

    power_pc = df_power.loc[df_power['Cell-Name']['Leistung [W]'] == checkup][cell_name]['Pc'].values[0]
    power_pd = df_power.loc[df_power['Cell-Name']['Leistung [W]'] == checkup][cell_name]['Pd'].values[0]

    dyn_cap_prof1 = df_dyn_cap.loc[df_dyn_cap['Cell-Name']['Kapazität [Ah]'] == checkup][cell_name]['Profil-1'].values[0]
    dyn_cap_prof2 = df_dyn_cap.loc[df_dyn_cap['Cell-Name']['Kapazität [Ah]'] == checkup][cell_name]['Profil-2'].values[0]

    tree = ET.parse(measurement)
    root = tree.getroot()
    # Extract device information
    device_info = root.find('.//deviceInformation')
    hardware_revision = device_info.find('hardware').get('revision')
    hardware_serial = device_info.find('hardware').get('serial')
    software_revision = device_info.find('software').get('revision')
    calibration_date = device_info.find('calibration').get('date')

    # Extract device configuration
    device_config = root.find('.//deviceConfiguration')
    current_excitation = device_config.find('currentExcitation')
    periods = current_excitation.get('periods')
    current_scaling_factor = current_excitation.get('currentScalingFactor')

    frequency_table = current_excitation.find('frequencyTable')
    frequency_count = frequency_table.get('frequencyCount')

    continuous_measurement = device_config.find('continousMeasurement')
    cm_period = continuous_measurement.get('period')

    enhancements = device_config.find('enhancements')
    discard_first_least_common_periods = enhancements.get('discardFirstLeastCommonPeriods')

    # Extract measurement results
    measurement_results = root.find('.//measurementResults')
    multiplexer_channel = measurement_results.get('multiplexerChannel')
    battery_name = measurement_results.get('batteryName')

    battery_voltage = float(measurement_results.find('batteryVoltage').get('voltage'))
    temperature_sensor = float(measurement_results.find('temperatureSensor').get('temperature'))

    impedance_spectrum = measurement_results.find('impedanceSpectrum')
    voltage_amplitude = impedance_spectrum.get('voltageAmplitude')
    clipped = impedance_spectrum.get('clipped')
    current_amplitude = impedance_spectrum.get('currentAmplitude')
    current_average = impedance_spectrum.get('currentAverage')
    timestamp_date = impedance_spectrum.get('date')
    timestamp_time = impedance_spectrum.get('time')
    
    # Combine date and time into the desired format
    timestamp = f"{timestamp_date.replace('/', '-')} {timestamp_time}"

    

    frequencies = []
    zRe_values = []
    zIm_values = []

    if impedance_spectrum is not None:
        # Iterate through all spectrumData elements within impedanceSpectrum
        for spectrum_data in impedance_spectrum.findall('spectrumData'):
            freq = float(spectrum_data.get('freq'))
            zRe = float(spectrum_data.get('zRe'))
            zIm = float(spectrum_data.get('zIm'))
            
            frequencies.append(freq)
            zRe_values.append(zRe)
            zIm_values.append(zIm)

    timestamp = [timestamp + ".00000"+str(i) for i in np.arange(0, len(frequencies))]
    
    # Create a DataFrame from the extracted data
    df = pd.DataFrame({
        'EIS_measurement_id' : [id] * len(frequencies),
        'EIS_Frequency': frequencies,
        'EIS_Z_abs': np.abs(np.array(zRe_values)+ 1j*np.array(zIm_values)),
        'EIS_Z_phase': np.angle(np.array(zRe_values)+ 1j*np.array(zIm_values)),
        'Capacity': [capacity] * len(frequencies),
        'EFC': [efc] * len(frequencies),
        'PowerPc': [power_pc] * len(frequencies),
        'PowerPd': [power_pd] * len(frequencies),
        'DynamicCapacityProfil1': [dyn_cap_prof1] * len(frequencies),
        'DynamicCapacityProfil2': [dyn_cap_prof2] * len(frequencies),
        'SOC': soc,
        'Checkup': checkup,
        'Temperature': temp,
        'Cell_Name': cell_name,
        'HardwareRevision': [hardware_revision] * len(frequencies),
        'HardwareSerial': [hardware_serial] * len(frequencies),
        'SoftwareRevision': [software_revision] * len(frequencies),
        'CalibrationDate': [calibration_date] * len(frequencies),
        'Periods': [periods] * len(frequencies),
        'CurrentScalingFactor': [current_scaling_factor] * len(frequencies),
        'FrequencyCount': [frequency_count] * len(frequencies),
        'ContinousMeasurementPeriod': [cm_period] * len(frequencies),
        'DiscardFirstLeastCommonPeriods': [discard_first_least_common_periods] * len(frequencies),
        'MultiplexerChannel': [multiplexer_channel] * len(frequencies),
        'BatteryName': [battery_name] * len(frequencies),
        'Voltage': [battery_voltage] * len(frequencies),
        'Temperature_Inspectrum': [temperature_sensor] * len(frequencies),
        'VoltageAmplitude': [voltage_amplitude] * len(frequencies),
        'Clipped': [clipped] * len(frequencies),
        'CurrentAmplitude': [current_amplitude] * len(frequencies),
        'CurrentAverage': [current_average] * len(frequencies),
        'Time': timestamp
    })
    
    return df

In [ ]:
df = pd.DataFrame()
for id, measurement in enumerate(measurements):
    df = pd.concat([df,extract_impedance_spectrum_to_df(measurement,id, df_capacity, df_power, df_dyn_cap, df_efc)],ignore_index=True)

In [ ]:
for battery_name in df.Cell_Name.unique():
    print(battery_name)
    df_cell = df.loc[df['Cell_Name'] == battery_name]
    df_cell.sort_values(['Time'], inplace=True)
    df_cell.reset_index(drop=True, inplace=True)
    df_cell.to_csv("./export/"+f"{battery_name}.csv", index=False)